# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. The target's baseline — settled here, because ML-06 could not settle it

Two different things get called a baseline in this assignment. Section 1 onward builds the **rule
baseline**: a hand-written score the model must beat. This section settles the other one first,
because nothing below can be evaluated without it.

**What ML-06 established.** `target = asinh(future_daily) − asinh(baseline_daily)` used the recent
30-day rate as `baseline_daily`. That quantity also drives the features, so a page with a large recent
window scores high on the signals *and* low on the target by arithmetic alone. Three separate
findings turned out to be that one mechanism:

| finding | what it looked like | what it was |
|---|---|---|
| Test 3 | a 7x decline gradient by peak ratio | 89% the label's own exclusion rule |
| simulation | pages reverting after a peak | a null with no future produced 79.0 points against 12.6 observed |
| Test 5 → Test 10 | signal hiding inside clients | a null produced ρ −0.66 against −0.09 observed |

**So the baseline must not appear in the features.** That is a testable property, not a matter of
taste, and the test already exists: build the target on a candidate baseline, then correlate the
signals against it using a **randomised future**. A null carrying no information about what happens
next should produce **ρ ≈ 0**. Any candidate where it does not is manufacturing the correlation.

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy scipy matplotlib python-dotenv

import os
import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

REPO = "FlyRank/internship-warehouse"
MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]
daily_files = [
    hf_hub_download(repo_id=REPO, repo_type="dataset",
                    filename=f"fact_content_daily_performance/month={m}/data_0.parquet", token=token)
    for m in MONTHS
]
con = duckdb.connect()
REL = "read_parquet([" + ", ".join(f"'{f}'" for f in daily_files) + "])"
D1 = "2026-03-31"

# Daily series for pages with enough pre-decision history to hold out a window.
q = f"""
WITH dense AS (
  SELECT content_hash_id FROM {REL}
  WHERE report_date < DATE '{D1}'
  GROUP BY 1 HAVING COUNT(*) FILTER (WHERE gsc_impressions > 0) >= 120)
SELECT f.content_hash_id, f.report_date, f.gsc_impressions
FROM {REL} f JOIN dense USING (content_hash_id)
WHERE f.report_date < DATE '{D1}' + INTERVAL 30 DAY
ORDER BY 1, 2"""
piv = con.sql(q).df().pivot(index="report_date", columns="content_hash_id",
                            values="gsc_impressions").fillna(0)
piv.index = pd.to_datetime(piv.index)

D_ts = pd.Timestamp(D1)
pre = piv[piv.index < D_ts]
fut = piv[(piv.index >= D_ts) & (piv.index < D_ts + pd.Timedelta(days=30))]
print(f"{piv.shape[1]:,} pages | {len(pre)} pre-decision days | {len(fut)} future days")


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


27,801 pages | 120 pre-decision days | 30 future days


**Four candidate baselines, chosen to span the trade-off.**

The tension is real: a baseline should reflect *where the page currently is*, which pushes toward
recent data — but recent data is what the features are built from. Each candidate resolves it
differently.

| candidate | definition | shares with features? |
|---|---|---|
| **A** recent 30d | the current design | fully — this is the control |
| **B** 90-day mean | contains the recent 30 | partly |
| **C** days 30–90 | the recent 30 held out entirely | no |
| **D** 90-day median | robust to spikes, still contains recent 30 | partly |

**C is the one to beat.** It costs a month of recency and gains complete separation. Whether that
trade is worth making is what the null decides, not preference.

In [2]:
rng = np.random.default_rng(7)

recent30 = pre.tail(30).mean().values          # most recent 30 days
older60 = pre.tail(90).head(60).mean().values  # days 30-90 back: recent month held out
full90 = pre.tail(90).mean().values
med90 = pre.tail(90).median().values
actual = fut.mean().values

# Null future: a random 30-day window from each page's OWN pre-decision history.
# Level and volatility preserved; all information about what happens next destroyed.
pre_arr = pre.values
starts = rng.integers(0, len(pre_arr) - 30, size=piv.shape[1])
null_fut = np.array([pre_arr[s:s + 30, i].mean() for i, s in enumerate(starts)])

safe = lambda n, d: np.divide(n, d, out=np.full_like(n, np.nan, dtype=float), where=d > 0)
signals = {
    "peak_ratio": safe(recent30, full90),
    "prior_trend": safe(recent30 - pre.tail(60).head(30).mean().values,
                        pre.tail(60).head(30).mean().values),
}
candidates = {"A recent 30d": recent30, "B 90-day mean": full90,
              "C days 30-90": older60, "D 90-day median": med90}

rows = []
for cname, base in candidates.items():
    t_obs = np.arcsinh(actual) - np.arcsinh(base)
    t_null = np.arcsinh(null_fut) - np.arcsinh(base)
    for sname, sig in signals.items():
        ok = np.isfinite(sig) & np.isfinite(t_obs) & np.isfinite(t_null)
        s = pd.Series(sig[ok])
        rows.append({
            "baseline": cname, "signal": sname, "n": int(ok.sum()),
            "rho_null": round(s.corr(pd.Series(t_null[ok]), method="spearman"), 4),
            "rho_observed": round(s.corr(pd.Series(t_obs[ok]), method="spearman"), 4),
        })

res = pd.DataFrame(rows)
res["artefact_share"] = (res["rho_null"].abs() /
                         res["rho_observed"].abs().replace(0, np.nan)).round(2)
print(res.to_string(index=False))
print()
print("rho_null is what a future carrying NO information produces.")
print("A baseline is independent of the features when rho_null is ~0.")

       baseline      signal     n  rho_null  rho_observed  artefact_share
   A recent 30d  peak_ratio 27801   -0.6827       -0.1485            4.60
   A recent 30d prior_trend 27801   -0.5977       -0.0409           14.61
  B 90-day mean  peak_ratio 27801   -0.2233        0.3066            0.73
  B 90-day mean prior_trend 27801   -0.2107        0.3666            0.57
   C days 30-90  peak_ratio 27801    0.1249        0.5057            0.25
   C days 30-90 prior_trend 27801    0.1058        0.5438            0.19
D 90-day median  peak_ratio 27801   -0.1986        0.3507            0.57
D 90-day median prior_trend 27801   -0.1227        0.4102            0.30

rho_null is what a future carrying NO information produces.
A baseline is independent of the features when rho_null is ~0.


**Verdict: candidate C. Holding out the recent month does not just shrink the artefact — it reverses
the finding.**

27,801 pages with ≥120 active pre-decision days:

| baseline | signal | ρ null | ρ observed | artefact share |
|---|---|---|---|---|
| **A** recent 30d | `peak_ratio` | **−0.6827** | −0.1485 | **4.60** |
| **A** recent 30d | `prior_trend` | **−0.5977** | −0.0409 | **14.61** |
| **B** 90-day mean | `peak_ratio` | −0.2233 | +0.3066 | 0.73 |
| **B** 90-day mean | `prior_trend` | −0.2107 | +0.3666 | 0.57 |
| **C** days 30–90 | `peak_ratio` | **+0.1249** | **+0.5057** | **0.25** |
| **C** days 30–90 | `prior_trend` | **+0.1058** | **+0.5438** | **0.19** |
| **D** 90-day median | `peak_ratio` | −0.1986 | +0.3507 | 0.57 |
| **D** 90-day median | `prior_trend` | −0.1227 | +0.4102 | 0.30 |

**Under A the artefact is 4.6 to 14.6 times larger than the signal. Under C the signal is 4 to 5
times larger than the artefact.** That ratio inverting is the whole point of the exercise.

**The sign flips, and that is the finding.** Under A, `peak_ratio` correlates **−0.15** with the
target: pages running hot appear to fall. Under C the same pages, the same futures, correlate
**+0.51**: pages running hot keep running above their older level. A was measuring change *from* the
hot window itself, so a hot window guaranteed a negative reading. Remove that and the relationship
points the other way.

This is consistent with everything ML-06 measured and could not interpret. Consecutive 30-day means
correlate at **0.799** — levels persist. A target built on the recent window fought that persistence;
a target built on an older window records it.

**Honest about the residual.** C's null is **+0.12** and **+0.11**, not zero. `peak_ratio` divides by
the 90-day mean, which still contains days 30–90, so a thread of shared history remains. It runs the
*same* direction as the observed signal now, so it inflates rather than inverts — and at roughly a
fifth of the magnitude. Reported rather than rounded away.

**What this settles.**

```
target = asinh(future_30d_daily_rate) - asinh(days_30_to_90_daily_rate)
```

The baseline is the page's own level over the 60 days ending one month before the decision point.
It costs a month of recency and buys separation from every feature built on the recent window.

**Caveat, same as Test 10's.** This runs on pages with ≥120 active days, where a baseline window can
actually be held out. Sparse pages have less history to spare and the residual may behave differently
there; section 2 checks the queue on the full cohort rather than assuming it carries over.

### The cohort, rebuilt on the settled target

`asinh` needs no non-zero denominator, so the two trend filters ML-06 was forced to apply — and which
Test 8 showed inflate the decline rate by 12.6 points — are no longer required. This cohort keeps
every page with any activity in the 90-day window, including the ones that fell to zero.

`baseline_daily` is the days 30–90 rate over 60 days. Where a page has no older history at all,
`asinh(0) = 0` and the target reduces to `asinh(future_daily)` — a well-defined number, not a
division by zero.

In [3]:
dim_file = hf_hub_download(repo_id=REPO, repo_type="dataset",
                           filename="dim_content.parquet", token=token)

q_cohort = f"""
WITH prior AS (
  SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
    SUM(gsc_impressions) AS impr_90d,
    SUM(gsc_clicks) AS clicks_90d,
    SUM(gsc_sum_position) AS sum_position_90d,
    SUM(gsc_impressions) FILTER (
        WHERE report_date < DATE '{D1}' - INTERVAL 30 DAY) AS older60_impr,
    SUM(gsc_impressions) FILTER (
        WHERE report_date >= DATE '{D1}' - INTERVAL 30 DAY) AS recent30_impr,
    SUM(ga4_sessions) AS ga4_sessions_90d,
    BOOL_OR(ga4_data_available) AS ga4_available
  FROM {REL}
  WHERE report_date >= DATE '{D1}' - INTERVAL 90 DAY AND report_date < DATE '{D1}'
  GROUP BY content_hash_id HAVING SUM(gsc_impressions) > 0),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS future_impr
  FROM {REL}
  WHERE report_date >= DATE '{D1}' AND report_date < DATE '{D1}' + INTERVAL 30 DAY
  GROUP BY content_hash_id)
SELECT p.*, COALESCE(f.future_impr, 0) AS future_impr
FROM prior p LEFT JOIN future f USING (content_hash_id)
ORDER BY p.content_hash_id"""

df = con.sql(q_cohort).df()
dim = con.sql(f"SELECT * FROM read_parquet('{dim_file}')").df().drop(columns=["client_hash_id"])
df = df.merge(dim, on="content_hash_id", how="left")

df[["older60_impr", "recent30_impr"]] = df[["older60_impr", "recent30_impr"]].fillna(0)
df["baseline_daily"] = df["older60_impr"] / 60
df["future_daily"] = df["future_impr"] / 30
df["recent_daily"] = df["recent30_impr"] / 30
df["target"] = np.arcsinh(df["future_daily"]) - np.arcsinh(df["baseline_daily"])
df["avg_position"] = df["sum_position_90d"] / df["impr_90d"].replace(0, np.nan) + 1
df["went_to_zero"] = df["future_daily"] == 0

print(f"cohort: {len(df):,} pages | {df['client_hash_id'].nunique()} clients")
print(f"  target: median {df['target'].median():.4f} | "
      f"declining (<0) {(df['target'] < 0).mean():.1%} | "
      f"growing (>0) {(df['target'] > 0).mean():.1%}")
print(f"  pages with no older history (baseline = 0): {(df['baseline_daily'] == 0).sum():,}")
print(f"  pages that went to zero: {df['went_to_zero'].sum():,} "
      f"({df['went_to_zero'].mean():.1%})")

cohort: 202,073 pages | 53 clients
  target: median 0.0663 | declining (<0) 42.3% | growing (>0) 53.9%
  pages with no older history (baseline = 0): 37,584
  pages that went to zero: 40,479 (20.0%)


### Before writing the rule: does every client have the columns the rule would use?

A rule that reads a column only some clients populate does not rank those clients — it ranks the
column's availability. ML-06 Test 6 showed missingness tracks `content_type`; this asks the sharper
question, whether any group is **structurally** without a field the rest have.

The measure is the spread of availability across groups. A field at 100% for one client and 0% for
another is not a feature, it is a client indicator wearing a feature's name.

In [4]:
candidate_fields = [c for c in ["search_volume", "keyword_difficulty", "cpc", "competition",
                                "word_count", "char_count", "backlinks", "referring_domains",
                                "content_created_date", "content_updated_date", "main_intent",
                                "ga4_sessions_90d", "impr_90d", "avg_position"]
                    if c in df.columns]


def availability(group_col, min_pages=200):
    big = df.groupby(group_col).filter(lambda g: len(g) >= min_pages)
    rows = []
    for f in candidate_fields:
        av = big.groupby(group_col)[f].apply(lambda s: s.notna().mean() * 100)
        rows.append({"field": f, "groups": len(av),
                     "min_%": round(av.min(), 1), "median_%": round(av.median(), 1),
                     "max_%": round(av.max(), 1), "spread": round(av.max() - av.min(), 1),
                     "groups_at_0%": int((av < 1).sum()),
                     "groups_at_100%": int((av > 99).sum())})
    return pd.DataFrame(rows).sort_values("spread", ascending=False)


for col in ["client_hash_id", "content_type"]:
    if col in df.columns:
        print("=" * 78)
        print(f"availability by {col} (groups with >= 200 pages)")
        print("=" * 78)
        print(availability(col).to_string(index=False))
        print()

availability by client_hash_id (groups with >= 200 pages)


               field  groups  min_%  median_%  max_%  spread  groups_at_0%  groups_at_100%
    ga4_sessions_90d      36    0.0     100.0  100.0   100.0             6              29
           backlinks      36    0.0      72.0  100.0   100.0             7              15
          word_count      36    5.2     100.0  100.0    94.8             0              26
          char_count      36    5.2     100.0  100.0    94.8             0              26
         competition      36    5.9      99.9  100.0    94.1             0              22
                 cpc      36    5.9      99.9  100.0    94.1             0              22
       search_volume      36    5.9      99.9  100.0    94.1             0              22
         main_intent      36    6.0      99.6  100.0    94.0             0              21
content_updated_date      36  100.0     100.0  100.0     0.0             0              36
content_created_date      36  100.0     100.0  100.0     0.0             0              36

               field  groups  min_%  median_%  max_%  spread  groups_at_0%  groups_at_100%
       search_volume       3    0.0      96.8  100.0   100.0             1               1
                 cpc       3    0.0      96.8  100.0   100.0             1               1
         competition       3    0.0      96.8  100.0   100.0             1               1
         main_intent       3    0.0      96.9  100.0   100.0             1               1
           backlinks       3    0.0      61.9   99.9    99.9             1               1
          word_count       3   67.5      98.9  100.0    32.5             0               1
          char_count       3   67.5      98.9  100.0    32.5             0               1
    ga4_sessions_90d       3   78.8      96.8  100.0    21.2             0               1
content_updated_date       3  100.0     100.0  100.0     0.0             0               3
content_created_date       3  100.0     100.0  100.0     0.0             0               3

**Verdict: yes — and only four fields are available to every client and every content type.**

By client (36 clients with ≥200 pages):

| field | min % | median % | max % | clients at 0% |
|---|---|---|---|---|
| `ga4_sessions_90d` | **0.0** | 100.0 | 100.0 | **6 of 36** |
| `backlinks` | **0.0** | 72.0 | 100.0 | **7 of 36** |
| `word_count` / `char_count` | 5.2 | 100.0 | 100.0 | 0 |
| `search_volume` / `cpc` / `competition` | 5.9 | 99.9 | 100.0 | 0 |
| `main_intent` | 6.0 | 99.6 | 100.0 | 0 |
| `impr_90d`, `avg_position`, both date fields | **100.0** | 100.0 | 100.0 | **0** |

By content type (3 types with ≥200 pages), one type has **0%** of `search_volume`, `cpc`,
`competition`, `main_intent` and `backlinks` — the entire keyword-and-links enrichment is absent for
a whole class of pages.

**Six clients have no GA4 at all. Seven have no backlinks data at all.** Not sparse — absent. A rule
reading either field would score 29 clients and silently rank 6 of them at the bottom regardless of
page health, which is a client indicator wearing a feature's name.

**What survives as universally available:** `impr_90d`, `avg_position`, `content_created_date`,
`content_updated_date`.

**And one of those four is out on other grounds.** ML-06 Finding 1 established that
`content_updated_date` is an export-time snapshot sitting *after* the decision point for 77.3% of
pages at D1. It leaks. That leaves **three fields the rule may legitimately use**:

```
impr_90d              how much traffic there is to lose
avg_position          where the page ranks
content_created_date  how old it is  (-> content_age_days)
```

That is a narrow base, and it is the honest one. It also happens to match the shape the
building-baselines skill suggests: *"it used to get traffic, it's getting old, and its position is
slipping."*

## 1. My rule and its reason codes

**In plain words.** A page is worth reviewing first if it is **losing ground against its own recent
past**, it still carries **enough traffic that losing it matters**, and it is **old enough that decay
is a plausible explanation** rather than a page that never started.

Three conditions, all from the three universally-available fields, no fitted weights:

| condition | test | why |
|---|---|---|
| `slipping` | recent 30-day rate below the days 30–90 rate | the page is trending down against its own history |
| `worth_saving` | `impr_90d >= 500` | below this there is little to protect; ML-06 Finding 3 showed the median dead page ran 0.13 impressions a day |
| `mature` | `content_age_days >= 180` | decay needs something to decay from |

**Score** = the size of the slip × the traffic at risk, gated on all three. Ranking on the *product*
rather than the slip alone means a 20% drop on a large page outranks a 60% drop on a small one, which
is the order a specialist with 100 slots actually wants.

**Reason codes** — every scored page carries why:

- `slipping_visible_mature` — all three conditions, the intended case
- `slipping_visible_new` — slipping and visible but under 180 days: real, but decay is the wrong story
- `slipping_thin` — slipping but under 500 impressions: probably noise
- `stable_or_growing` — not slipping; scored zero
- `no_older_baseline` — no days 30–90 history to compare against; cannot be judged by this rule

**Dead pages do not enter this queue.** Per ML-06 Finding 3 they carry a separate volume-gated flag,
ranked by prior daily rate. A page at zero is an emergency, not a ranking problem.

In [5]:
MIN_IMPRESSIONS = 500
MIN_AGE_DAYS = 180
DECISION = pd.Timestamp(D1)

r = df.copy()
r["content_age_days"] = (DECISION - pd.to_datetime(r["content_created_date"])).dt.days

# The slip: how far the recent 30 days sit below the days 30-90 baseline.
# Positive means falling. Uses only fact-table columns available for every client.
r["slip"] = np.where(r["baseline_daily"] > 0,
                     (r["baseline_daily"] - r["recent_daily"]) / r["baseline_daily"],
                     np.nan)

slipping = r["slip"] > 0
worth_saving = r["impr_90d"] >= MIN_IMPRESSIONS
mature = r["content_age_days"] >= MIN_AGE_DAYS

r["reason_code"] = np.select(
    [r["baseline_daily"] == 0,
     slipping & worth_saving & mature,
     slipping & worth_saving & ~mature,
     slipping & ~worth_saving],
    ["no_older_baseline", "slipping_visible_mature", "slipping_visible_new", "slipping_thin"],
    default="stable_or_growing")

# score = size of the slip x traffic at risk, gated on all three conditions
r["baseline_score"] = np.where(
    slipping & worth_saving & mature,
    r["slip"].clip(lower=0) * r["impr_90d"],
    0.0)

print(r["reason_code"].value_counts().to_string())
print()
print(f"pages with a non-zero score: {(r['baseline_score'] > 0).sum():,} "
      f"({(r['baseline_score'] > 0).mean():.1%} of the cohort)")

# the separate flag, per ML-06 Finding 3
flag = r["went_to_zero"] & (r["recent_daily"] >= 1)
r["dead_page_flag"] = flag
print(f"dead-page flag (zero future AND >= 1 impr/day recent): {int(flag.sum()):,} pages "
      f"over {r.loc[flag, 'client_hash_id'].nunique()} clients")

reason_code
stable_or_growing          99473
slipping_thin              43729
no_older_baseline          37584
slipping_visible_mature    16727
slipping_visible_new        4560

pages with a non-zero score: 16,727 (8.3% of the cohort)
dead-page flag (zero future AND >= 1 impr/day recent): 1,713 pages over 38 clients


## 2. Build the ranked queue (writes the CSV)

Ranked **per client at K = 100**, monthly, per the scope settled in ML-03 §3 — pooled across clients
so every page counts once, which is the correction ML-05 needed.

**The evaluation cut, stated rather than hidden.** The target is continuous, so "actually declined"
is an evaluation parameter now: `target < 0`, meaning the page's future 30-day rate fell below its
days 30–90 baseline. Base rate on this cohort is **42.3%**.

**Three comparisons, because a precision number alone means nothing:**

| comparison | what it tests |
|---|---|
| **base rate** | what random picking gives |
| **shuffled target** | whether the ordering carries any information at all |
| **persistence null** — future replaced by the recent 30-day rate | whether the rule merely restates the slip it is built from |

The third is the one that matters. `slip` compares the recent window against the days 30–90 baseline,
and the target compares the *future* against that same baseline. If the rule scores as well when the
future is replaced by the present, it is describing what already happened, not predicting.

In [6]:
from pathlib import Path

rng2 = np.random.default_rng(2)
K = 100
r["declined"] = r["target"] < 0


def precision_at_k_per_client(frame, score_col, label_col, k=K):
    """Pooled: total hits over total picks, so every page counts once."""
    hits = picks = 0
    for _, g in frame.groupby("client_hash_id"):
        g = g.nlargest(min(k, len(g)), score_col)
        hits += int(g[label_col].sum())
        picks += len(g)
    return hits / picks if picks else np.nan


scored = r[r["baseline_score"] > 0].copy()
base_rate = r["declined"].mean()

# null 1: shuffle the label inside each client -- destroys the pairing, keeps marginals
scored["declined_shuffled"] = (scored.groupby("client_hash_id")["declined"]
                               .transform(lambda s: rng2.permutation(s.values)))
# null 2: persistence -- the future is simply the recent 30-day rate
scored["declined_persist"] = (np.arcsinh(scored["recent_daily"])
                              - np.arcsinh(scored["baseline_daily"])) < 0

rows = [
    {"ranking": "the rule (baseline_score)", "label": "real future",
     "precision_at_100": precision_at_k_per_client(scored, "baseline_score", "declined")},
    {"ranking": "random order", "label": "real future",
     "precision_at_100": precision_at_k_per_client(
         scored.assign(rand=rng2.random(len(scored))), "rand", "declined")},
    {"ranking": "the rule", "label": "shuffled in client",
     "precision_at_100": precision_at_k_per_client(scored, "baseline_score", "declined_shuffled")},
    {"ranking": "the rule", "label": "persistence null",
     "precision_at_100": precision_at_k_per_client(scored, "baseline_score", "declined_persist")},
]
# The lift that matters is against the pool the rule actually ranks, not the
# full cohort. Ranking only ever reorders pages the filter already selected,
# so crediting the filter's selection to the ranking repeats ML-05's mistake.
pool_rate = scored["declined"].mean()

ev = pd.DataFrame(rows)
ev["precision_at_100"] = ev["precision_at_100"].round(4)
ev["lift_vs_cohort"] = (ev["precision_at_100"] / base_rate).round(2)
ev["lift_vs_pool"] = (ev["precision_at_100"] / pool_rate).round(3)

print(f"base rate, full cohort ({len(r):,} pages, {r['client_hash_id'].nunique()} clients): "
      f"{base_rate:.4f}")
print(f"base rate, the pool the rule ranks ({len(scored):,} pages, "
      f"{scored['client_hash_id'].nunique()} clients): {pool_rate:.4f}")
print()
print(ev.to_string(index=False))
print()

# Coverage: a per-client queue is worthless to a client it never fills.
per_client = r.groupby("client_hash_id")["baseline_score"].apply(lambda s: (s > 0).sum())
print(f"clients with at least one scored page: {(per_client > 0).sum()} of {len(per_client)}")
print(f"clients with a full K={K} queue:        {(per_client >= K).sum()} of {len(per_client)}")
print(f"pages scored per client: median {per_client.median():.0f}, max {per_client.max():,}")

# ---- write the queue -------------------------------------------------------
out_cols = ["content_hash_id", "client_hash_id", "baseline_score", "reason_code",
            "slip", "impr_90d", "avg_position", "content_age_days",
            "recent_daily", "baseline_daily", "dead_page_flag"]
queue = (r[r["baseline_score"] > 0]
         .sort_values(["client_hash_id", "baseline_score"], ascending=[True, False])
         .groupby("client_hash_id").head(K)[out_cols])
outdir = Path("../outputs")
outdir.mkdir(parents=True, exist_ok=True)
queue.to_csv(outdir / "baseline_action_score.csv", index=False)
print()
print(f"wrote {outdir / 'baseline_action_score.csv'}: {len(queue):,} rows, "
      f"{queue['client_hash_id'].nunique()} clients")

base rate, full cohort (202,073 pages, 53 clients): 0.4228
base rate, the pool the rule ranks (16,727 pages, 23 clients): 0.8037

                  ranking              label  precision_at_100  lift_vs_cohort  lift_vs_pool
the rule (baseline_score)        real future            0.9427            2.23         1.173
             random order        real future            0.8854            2.09         1.102
                 the rule shuffled in client            0.8681            2.05         1.080
                 the rule   persistence null            1.0000            2.37         1.244

clients with at least one scored page: 23 of 53
clients with a full K=100 queue:        12 of 53
pages scored per client: median 0, max 4,425

wrote ..\outputs\baseline_action_score.csv: 1,501 rows, 23 clients


**Verdict: the rule detects pages that have already declined. It does not predict decline, and it
does not reach most clients.**

| ranking | label | P@100 | lift vs cohort | lift vs pool |
|---|---|---|---|---|
| **the rule** | real future | **0.9427** | 2.23 | **1.173** |
| random order | real future | 0.8854 | 2.09 | 1.102 |
| the rule | shuffled in client | 0.8681 | 2.05 | 1.080 |
| the rule | persistence null | **1.0000** | 2.37 | 1.244 |

**The 2.23x is not a real lift.** It divides precision measured on the pool the rule ranks by the
*full cohort's* base rate. The pool declines at **80.4%** against the cohort's 42.3% — the filter
already selected pages that were falling. Crediting that selection to the ranking is the same error
that made per-client recall read 48.5% in ML-05. Against its own pool the rule lifts **1.173x**.

**The ranking contributes little.** Random ordering *within the same pool* scores 0.8854 against the
rule's 0.9427. The ordering is worth about **six points of precision** — real, but small next to what
the filter does.

**The persistence null returns 1.0000, and that is the finding.** `slipping` means recent < baseline;
`declined` means future < baseline. Both measured against the same days 30–90 window. With 30-day
levels autocorrelated at 0.799, a page already below its baseline mostly stays below it, so the rule
scores perfectly against a future that is simply the present. **It is a detector of pages that have
already fallen, betting on persistence — not a predictor of pages about to fall.**

That distinction matters to a specialist. Told these pages are "at risk", they would expect to
prevent something. What the queue actually offers is a list of pages that already lost ground and
will probably keep losing it — useful, but a different promise.

**Coverage fails outright.**

| | |
|---|---|
| clients with at least one scored page | **23 of 53** |
| clients with a full K = 100 queue | **12 of 53** |
| pages scored per client | **median 0**, max 4,425 |

**The median client receives an empty queue.** For a per-client product that is not a shortfall, it is
a non-delivery. `MIN_IMPRESSIONS = 500` is the likely cause — it came from `w01`'s gut-check on the
starter CSV and has never been tested against anything.

**Kept rather than fixed in place.** The building-baselines skill says a baseline's job is to be
honestly beatable, and this one now has three measured weaknesses for ML-08 to beat: it cannot rank
what it has not filtered, it restates the present rather than predicting the future, and it reaches
23 of 53 clients. A revision follows in section 4 rather than overwriting this, so the two can be
compared.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.